{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "header"
   },
   "source": [
    "# 🚗 Automatic Number Plate Recognition (ANPR) System\n",
    "## Detect and read license plates from images using YOLO + EasyOCR\n",
    "\n",
    "### 🔧 Setup - Run this cell first"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "setup"
   },
   "source": [
    "# Install required packages\n",
    "!pip install ultralytics easyocr opencv-python matplotlib -q\n",
    "\n",
    "from ultralytics import YOLO\n",
    "import cv2\n",
    "import easyocr\n",
    "from google.colab import files\n",
    "import numpy as np\n",
    "import os\n",
    "import re\n",
    "\n",
    "print(\"✅ All packages installed successfully!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "load_model"
   },
   "source": [
    "### 📥 Download Detection Model\n",
    "This downloads the pre-trained YOLO model for license plate detection"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "download_model"
   },
   "source": [
    "# Download the pre-trained model\n",
    "!wget https://github.com/PurplePegasus08/ANPD-using-YOLO/raw/main/best.pt -O best.pt\n",
    "\n",
    "import os\n",
    "if os.path.exists('best.pt'):\n",
    "    size = os.path.getsize('best.pt') / (1024*1024)\n",
    "    print(f\"✅ Model downloaded! Size: {size:.2f} MB\")\n",
    "else:\n",
    "    print(\"❌ Download failed - trying backup...\")\n",
    "    !wget https://huggingface.co/makhresearch/persian-license-plate-detector/resolve/main/best.pt -O best.pt"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "preprocessing"
   },
   "source": [
    "### 🔧 Image Preprocessing Functions\n",
    "These functions enhance images for better OCR results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "preprocess"
   },
   "source": [
    "def enhance_plate_for_ocr(plate_img):\n",
    "    \"\"\"Enhance plate image for better OCR\"\"\"\n",
    "    if len(plate_img.shape) == 3:\n",
    "        gray = cv2.cvtColor(plate_img, cv2.COLOR_RGB2GRAY)\n",
    "    else:\n",
    "        gray = plate_img\n",
    "    \n",
    "    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)\n",
    "    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))\n",
    "    enhanced = clahe.apply(gray)\n",
    "    denoised = cv2.medianBlur(enhanced, 3)\n",
    "    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)\n",
    "    return binary\n",
    "\n",
    "def clean_plate_text(text):\n",
    "    \"\"\"Clean and format license plate text\"\"\"\n",
    "    cleaned = re.sub(r'[^A-Z0-9\\s]', '', text.upper())\n",
    "    return ' '.join(cleaned.split())\n",
    "\n",
    "print(\"✅ Preprocessing functions loaded!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "load_models"
   },
   "source": [
    "### 🚀 Load YOLO and EasyOCR Models"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "models"
   },
   "source": [
    "print(\"=\"*50)\n",
    "print(\"Loading License Plate Detector...\")\n",
    "detector = YOLO('best.pt')\n",
    "\n",
    "print(\"Loading EasyOCR for text reading...\")\n",
    "reader = easyocr.Reader(['en'])\n",
    "print(\"✅ Models loaded!\")\n",
    "print(\"=\"*50)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "upload"
   },
   "source": [
    "### 📸 Upload Your Image\n",
    "Click the button below and select a car/plate image"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "process"
   },
   "source": [
    "print(\"\\n📁 Upload your license plate image:\")\n",
    "uploaded = files.upload()\n",
    "\n",
    "for img_name, img_data in uploaded.items():\n",
    "    print(f\"\\n{'─'*50}\")\n",
    "    print(f\"Processing: {img_name}\")\n",
    "    \n",
    "    # Save and read image\n",
    "    with open(img_name, 'wb') as f:\n",
    "        f.write(img_data)\n",
    "    \n",
    "    img = cv2.imread(img_name)\n",
    "    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\n",
    "    \n",
    "    # Detect plates\n",
    "    results = detector(img_rgb, conf=0.25)\n",
    "    \n",
    "    if results[0].boxes and len(results[0].boxes) > 0:\n",
    "        print(f\"✓ Found {len(results[0].boxes)} potential plate region(s)\")\n",
    "        \n",
    "        all_plate_texts = []\n",
    "        \n",
    "        for box in results[0].boxes:\n",
    "            confidence = float(box.conf[0])\n",
    "            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())\n",
    "            plate_crop = img_rgb[y1:y2, x1:x2]\n",
    "            \n",
    "            if plate_crop.size > 0:\n",
    "                enhanced_plate = enhance_plate_for_ocr(plate_crop)\n",
    "                ocr_result = reader.readtext(enhanced_plate, detail=0, paragraph=False)\n",
    "                \n",
    "                if ocr_result:\n",
    "                    plate_text = clean_plate_text(ocr_result[0])\n",
    "                    if len(plate_text) >= 3:\n",
    "                        all_plate_texts.append({\n",
    "                            'text': plate_text,\n",
    "                            'confidence': confidence,\n",
    "                            'bbox': (x1, y1, x2, y2)\n",
    "                        })\n",
    "        \n",
    "        if all_plate_texts:\n",
    "            all_plate_texts.sort(key=lambda x: len(x['text']), reverse=True)\n",
    "            \n",
    "            print(\"\\n\" + \"=\"*50)\n",
    "            print(\"📋 LICENSE PLATE RESULTS\")\n",
    "            print(\"=\"*50)\n",
    "            \n",
    "            for i, plate in enumerate(all_plate_texts[:3], 1):\n",
    "                print(f\"\\n  PLATE #{i}: {plate['text']}\")\n",
    "                print(f\"  Confidence: {plate['confidence']:.2f}\")\n",
    "            \n",
    "            print(\"\\n\" + \"=\"*50)\n",
    "            \n",
    "            # Save best plate\n",
    "            best = all_plate_texts[0]\n",
    "            x1, y1, x2, y2 = best['bbox']\n",
    "            plate_crop = img_rgb[y1:y2, x1:x2]\n",
    "            cv2.imwrite(f\"cropped_plate_{img_name}\", cv2.cvtColor(plate_crop, cv2.COLOR_RGB2BGR))\n",
    "            print(f\"\\n💾 Cropped plate saved: cropped_plate_{img_name}\")\n",
    "        else:\n",
    "            print(\"\\n⚠️ Could not read plate text\")\n",
    "    else:\n",
    "        print(\"❌ No license plate detected\")\n",
    "        print(\"   Tips: Use clear, well-lit images with visible plates\")\n",
    "    \n",
    "    os.remove(img_name)\n",
    "\n",
    "print(\"\\n✨ Processing complete!\")"
   ]
  }
 ],
 "metadata": {
  "accelerator": "GPU",
  "colab": {
   "provenance": []
  },
  "kernelspec": {
   "display_name": "Python 3",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 0
}